# 01_mlp_numpy.ipynb

Ce notebook explique pas à pas les concepts de base des réseaux de neurones et montre des implémentations pédagogiques en NumPy :
- perceptron
- régression logistique vectorisée
- MLP (1-2 couches) with forward et backward
- gradient checking

Les explications sont en français et le code est commenté pour un public débutant en Python.


In [ ]:
# Imports de base
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
np.random.seed(42)


## 1) Perceptron (algorithme simple)
On commence par un perceptron binaire (algorithme de Rosenblatt). C'est un très bon point de départ pour comprendre la logique du neurone.


In [ ]:
class Perceptron:
    def __init__(self, dim, lr=0.1, n_iter=1000):
        self.w = np.zeros(dim)
        self.b = 0.0
        self.lr = lr
        self.n_iter = n_iter

    def predict(self, X):
        # sortie 0/1
        z = X.dot(self.w) + self.b
        return (z >= 0).astype(int)

    def fit(self, X, y):
        for _ in range(self.n_iter):
            preds = self.predict(X)
            for xi, yi, pi in zip(X, y, preds):
                if pi != yi:
                    update = self.lr * (yi - pi)
                    self.w += update * xi
                    self.b += update


In [ ]:
# Test rapide sur données 'moons'
X, y = make_moons(noise=0.2, random_state=0)
p = Perceptron(dim=2, lr=0.1, n_iter=100)
p.fit(X, y)

# affichage frontière de décision
def plot_decision_boundary(model, X, y, title=''):
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X[:,0], X[:,1], c=y, s=40, cmap='viridis')
    plt.title(title)
    plt.show()

plot_decision_boundary(p, X, y, title='Perceptron sur moons')


## 2) Régression logistique (vectorisée)
La régression logistique est un modèle linéaire mais optimisé avec une loss log-sigmoïde (cross-entropy). On va l'implémenter de manière vectorisée et utiliser gradient descent.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_loss_and_grad(w, b, X, y):
    # w: (d,), X: (n,d), y: (n,) avec 0/1
    n = X.shape[0]
    z = X.dot(w) + b
    probs = sigmoid(z)
    # loss moyenne (cross-entropy)
    eps = 1e-12
    loss = -np.mean(y * np.log(probs+eps) + (1-y) * np.log(1-probs+eps))
    # gradient
    dz = probs - y
    dw = X.T.dot(dz) / n
    db = np.mean(dz)
    return loss, dw, db

# Entraînement simple
def train_logistic(X, y, lr=0.5, n_iter=2000):
    d = X.shape[1]
    w = np.zeros(d)
    b = 0.0
    losses = []
    for i in range(n_iter):
        loss, dw, db = logistic_loss_and_grad(w, b, X, y)
        w -= lr * dw
        b -= lr * db
        losses.append(loss)
    return w, b, losses

# test
Xs, ys = make_moons(noise=0.2, random_state=1)
w, b, losses = train_logistic(Xs, ys, lr=0.5, n_iter=2000)
plt.plot(losses)
plt.title('Loss during logistic training')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.show()

class LogisticModel:
    def __init__(self, w, b):
        self.w = w
        self.b = b
    def predict(self, X):
        return (sigmoid(X.dot(self.w) + self.b) >= 0.5).astype(int)

plot_decision_boundary(LogisticModel(w,b), Xs, ys, title='Régression logistique sur moons')


## 3) MLP from-scratch (NumPy) avec backprop
Nous implémenterons un MLP simple (une couche cachée) et coderons explicitement forward et backward. L'objectif est de comprendre la chaîne de dérivation (chain rule).


In [ ]:
def relu(z):
    return np.maximum(0, z)
def relu_grad(z):
    return (z > 0).astype(float)

def softmax(z):
    exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

def cross_entropy_loss_and_grad_logits(logits, y_onehot):
    # logits: (n, C), y_onehot: (n, C)
    n = logits.shape[0]
    probs = softmax(logits)
    eps = 1e-12
    loss = -np.sum(y_onehot * np.log(probs + eps)) / n
    grad = (probs - y_onehot) / n
    return loss, grad

class SimpleMLP:
    def __init__(self, input_dim, hidden_dim, output_dim):
        # initialisation simple (Xavier)
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2. / (input_dim + hidden_dim))
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2. / (hidden_dim + output_dim))
        self.b2 = np.zeros(output_dim)

    def forward(self, X):
        # renvoie logits et caches pour backward
        z1 = X.dot(self.W1) + self.b1
        a1 = relu(z1)
        logits = a1.dot(self.W2) + self.b2
        cache = (X, z1, a1)
        return logits, cache

    def backward(self, logits, cache, y_onehot):
        X, z1, a1 = cache
        loss, dlogits = cross_entropy_loss_and_grad_logits(logits, y_onehot)
        # dW2, db2
        dW2 = a1.T.dot(dlogits)
        db2 = np.sum(dlogits, axis=0)
        # propagate to hidden
        da1 = dlogits.dot(self.W2.T)
        dz1 = da1 * relu_grad(z1)
        dW1 = X.T.dot(dz1)
        db1 = np.sum(dz1, axis=0)
        grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}
        return loss, grads

    def update(self, grads, lr=1e-2):
        self.W1 -= lr * grads['W1']
        self.b1 -= lr * grads['b1']
        self.W2 -= lr * grads['W2']
        self.b2 -= lr * grads['b2']

    def predict(self, X):
        logits, _ = self.forward(X)
        return np.argmax(softmax(logits), axis=1)


In [ ]:
# Préparer un petit sous-ensemble de MNIST (ou digits de sklearn pour rapidité)
digits = load_digits()
X = digits.data / 16.0  # normalisation simple
y = digits.target
# pour l'exemple, restreindre à 3 classes pour simplicité
mask = y < 3
X = X[mask]
y = y[mask]
y_onehot = np.eye(3)[y]
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=0)

model = SimpleMLP(input_dim=X.shape[1], hidden_dim=64, output_dim=3)
losses = []
for epoch in range(200):
    logits, cache = model.forward(X_train)
    loss, grads = model.backward(logits, cache, y_train)
    model.update(grads, lr=0.01)
    losses.append(loss)
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, loss={loss:.4f}')
plt.plot(losses)
plt.title('Training loss (MLP NumPy)')
plt.show()

preds = model.predict(X_test)
acc = (preds == np.argmax(y_test, axis=1)).mean()
print('Accuracy on test (MLP NumPy) :', acc)


## 4) Gradient checking (vérification numérique)
Il est essentiel, surtout quand on implémente le backward soi-même, de vérifier que les gradients analytiques sont corrects en les comparant à des approximations numériques.


In [ ]:
def numerical_grad(model, X, y_onehot, param_name, eps=1e-5):
    # param_name in ['W1','b1','W2','b2']
    param = getattr(model, param_name)
    num_grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = param[idx]
        param[idx] = old_val + eps
        logits_plus, _ = model.forward(X)
        loss_plus, _ = cross_entropy_loss_and_grad_logits(logits_plus, y_onehot)
        param[idx] = old_val - eps
        logits_minus, _ = model.forward(X)
        loss_minus, _ = cross_entropy_loss_and_grad_logits(logits_minus, y_onehot)
        num_grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        param[idx] = old_val
        it.iternext()
    return num_grad

# Test gradient check sur un petit sous-ensemble pour rapidité
X_small = X_train[:10]
y_small = y_train[:10]
logits, cache = model.forward(X_small)
loss, grads = model.backward(logits, cache, y_small)
num_W1 = numerical_grad(model, X_small, y_small, 'W1')
# comparer nombre vs analytique (norme relative)
rel_err = np.linalg.norm(num_W1 - grads['W1']) / (np.linalg.norm(num_W1) + np.linalg.norm(grads['W1']) + 1e-12)
print('relative error W1:', rel_err)

# règle empirique: erreur relative < 1e-6 or 1e-7 indique bonne implémentation; si grand, debug shapes/derivatives


---
Ce notebook est conçu pour être exécuté pas à pas. Après l'avoir parcouru, tu auras une compréhension pratique de la rétropropagation et tu pourras réimplémenter la même architecture en PyTorch (notebook suivant).
